CELL 1: DATA LOAD 


In [ ]:
import pandas as pd
import numpy as np
import os

# Check input directory structure
input_path = '/kaggle/input/comment-category-prediction-challenge/'
print("Files in directory:", os.listdir(input_path))

# Load training and test datasets
train_data = pd.read_csv(os.path.join(input_path, 'train.csv'))
test_data = pd.read_csv(os.path.join(input_path, 'test.csv'))

# Basic text cleaning: handle missing values and normalize whitespace/case
# We use 'missing' for NaNs to ensure the vectorizers have a valid string to process
train_data['comment'] = train_data['comment'].fillna("missing").astype(str).str.lower().str.strip()
test_data['comment'] = test_data['comment'].fillna("missing").astype(str).str.lower().str.strip()

# Fill remaining missing numerical values with 0
train_data = train_data.fillna(0)
test_data = test_data.fillna(0)

# Feature engineering: Character length and upvote ratios
train_data['comment_len'] = train_data['comment'].str.len()
test_data['comment_len'] = test_data['comment'].str.len()

# Calculate the ratio of positive engagement relative to total votes
train_data['engagement_score'] = train_data['upvote'] / (train_data['upvote'] + train_data['downvote'] + 1)
test_data['engagement_score'] = test_data['upvote'] / (test_data['upvote'] + test_data['downvote'] + 1)

# Select numerical columns for the tabular part of the model
numeric_features = [
    'post_id', 'upvote', 'downvote', 'if_1', 'if_2', 
    'emoticon_1', 'emoticon_2', 'emoticon_3', 
    'comment_len', 'engagement_score'
]

X_numeric_train = train_data[numeric_features].values.astype(float)
X_numeric_test = test_data[numeric_features].values.astype(float)

print(f"Dataset summary: Train {train_data.shape}, Test {test_data.shape}")
print("Label distribution in training set:")
print(train_data['label'].value_counts().sort_index())

**Experimental Log**

v1 – v4: Initial Baselines
* Explored the dataset and established basic benchmarks using simple Logistic Regression on word-level TF-IDF.

v5 – v8: Feature Engineering & Tabular Integration
* Integrated numerical metadata (upvotes, downvotes, post_ids).
* Added custom features like comment_len and engagement_score.

v9 – v11: Model Diversification
* Experimented with non-linear models including LightGBM to capture engagement patterns.
* Tested Character-level n-grams (3-5) to handle misspellings and toxic text variations.
v12: Preliminary Stacking
* Combined model outputs using simple averaging, which improved the score but lacked robustness against data leakage.

v13-v15:(Current Final Version): 5-Fold CV Ensemble Stacker
* Implemented Stratified 5-Fold Cross-Validation to generate Out-of-Fold (OOF) predictions.
* Expert Layer: Logistic Regression (Text Linear), LightGBM (Tabular Non-linear), and Multinomial Naive Bayes (Frequency Diversity).
* Meta-Learner Layer: A tuned Multi-Layer Perceptron (MLP) stacker that learns the optimal weight for each expert.
* Result: Achieved the final competition cutoff of 0.80+ with significantly improved generalization.


CELL 1a: Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Target Distribution
plt.figure(figsize=(8, 5))
sns.countplot(x='label', data=train_data, palette='viridis')
plt.title('Distribution of Comment Categories')
plt.xlabel('Category Label')
plt.ylabel('Number of Comments')
plt.show()

# 2. Text Length Analysis by Category
plt.figure(figsize=(10, 6))
sns.boxplot(x='label', y='comment_len', data=train_data)
plt.title('Comment Length Distribution per Category')
plt.ylim(0, 1000) # Capping y-axis to see the boxes more clearly
plt.show()

# 3. Correlation Heatmap of Numerical Features
plt.figure(figsize=(10, 8))
numeric_cols = ['upvote', 'downvote', 'comment_len', 'engagement_score', 'label']
sns.heatmap(train_data[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Metadata Features')
plt.show()

# 4. Statistical Summary
print("Top 5 most engaged comments (Highest Upvotes):")
print(train_data.nlargest(5, 'upvote')[['comment', 'upvote', 'label']])

# Identification of Feature Types for Checklist
categorical_features = train_data.select_dtypes(include=['object']).columns.tolist()
numerical_features = train_data.select_dtypes(exclude=['object']).columns.tolist()

print(f"Categorical Features: {categorical_features}")
print(f"Numerical Features: {numerical_features}")

# Visualizing Missing Data (Mandatory Checklist Item)
plt.figure(figsize=(10,4))
sns.heatmap(train_data.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Data Check (Yellow indicates missing)')
plt.show()

| Component | Logistic Regression (Expert 1) | LightGBM (Expert 2) | MLP Stacker (Meta-Model) |
|---|---|---|---|
| **Data ($X, y$)** | Sparse TF-IDF (Word + Char n-grams) | Dense Metadata + Identity Categories | OOF Probability Predictions from Experts |
| **Model ($f$)** | Generalized Linear Model (Softmax) | Gradient Boosted Decision Trees (GBDT) | Multi-Layer Perceptron (Backprop Neural Net) |
| **Loss Function** | Categorical Cross-Entropy | Multi-class Log Loss | Categorical Cross-Entropy |
| **Optimization** | L-BFGS (Quasi-Newton Method) | GOSS (Gradient-based One-Side Sampling) | Adam (Stochastic Gradient Descent) |
| **Evaluation** | Macro F1-Score | Macro F1-Score | Leaderboard Macro F1 / Confusion Matrix |

CELL 2 MODELLING

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix, f1_score
import lightgbm as lgb
from scipy.sparse import hstack

# Suppress annoying warnings for clean output
warnings.filterwarnings('ignore')

# 1. Identity Feature Preprocessing
id_features = ['male', 'female', 'transgender', 'white', 'black', 'muslim', 'hindu']
for col in id_features:
    if col in train_data.columns:
        train_data[col] = train_data[col].fillna(0).astype(float)
        test_data[col] = test_data[col].fillna(0).astype(float)
    else:
        train_data[col], test_data[col] = 0.0, 0.0

# 2. Text Vectorization (Hybrid Word & Char n-grams)
# Pruned slightly for speed while keeping 'Hate' detection patterns
word_vec = TfidfVectorizer(max_features=25000, ngram_range=(1, 2), sublinear_tf=True, stop_words='english')
char_vec = TfidfVectorizer(max_features=15000, ngram_range=(3, 5), sublinear_tf=True, analyzer='char')

train_tfidf = hstack([word_vec.fit_transform(train_data['comment']), 
                      char_vec.fit_transform(train_data['comment'])])
test_tfidf = hstack([word_vec.transform(test_data['comment']), 
                     char_vec.transform(test_data['comment'])])

# 3. Metadata Extraction
def get_custom_features(df):
    # Punctuation rate calculation
    df['punc_rate'] = df['comment'].apply(lambda x: (str(x).count('!') + str(x).count('?')) / (len(str(x)) + 1))
    cols = ['if_1', 'if_2', 'upvote', 'downvote', 'comment_len', 'punc_rate'] + id_features
    return df[cols].fillna(0)

scaler = StandardScaler()
train_meta = scaler.fit_transform(get_custom_features(train_data))
test_meta = scaler.transform(get_custom_features(test_data))

# 4. Cross-Validation Pipeline
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y = train_data['label']

# Storing OOF predictions for the stacker
oof_lr, oof_lgbm, oof_nb = [np.zeros((len(train_data), 4)) for _ in range(3)]
test_lr, test_lgbm, test_nb = [np.zeros((len(test_data), 4)) for _ in range(3)]

print("Starting Expert Model Pipeline...")
for fold, (t_idx, v_idx) in enumerate(skf.split(train_data, y)):
    print(f"Working on Fold {fold + 1}...")
    
    # Expert 1: Logistic Regression (Text Focus)
    m1 = LogisticRegression(C=1.5, max_iter=500, n_jobs=-1)
    m1.fit(train_tfidf.tocsr()[t_idx], y.iloc[t_idx])
    oof_lr[v_idx] = m1.predict_proba(train_tfidf.tocsr()[v_idx])
    test_lr += m1.predict_proba(test_tfidf) / 5
    
    # Expert 2: LightGBM (Metadata Focus)
    # n_estimators reduced slightly for speed; learning rate adjusted to maintain accuracy
    m2 = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.04, num_leaves=31, random_state=42, verbosity=-1)
    m2.fit(train_meta[t_idx], y.iloc[t_idx])
    oof_lgbm[v_idx] = m2.predict_proba(train_meta[v_idx])
    test_lgbm += m2.predict_proba(test_meta) / 5

    # Expert 3: Naive Bayes (Frequency Baseline)
    m3 = MultinomialNB(alpha=0.5)
    m3.fit(train_tfidf.tocsr()[t_idx], y.iloc[t_idx])
    oof_nb[v_idx] = m3.predict_proba(train_tfidf.tocsr()[v_idx])
    test_nb += m3.predict_proba(test_tfidf) / 5

# 5. Stacking Layer with Grid Search
X_stack_train = np.hstack([oof_lr, oof_lgbm, oof_nb])
X_stack_test = np.hstack([test_lr, test_lgbm, test_nb])

print("Tuning Meta-Model (MLP Stacker)...")
mlp_params = {'alpha': [0.01, 0.1], 'learning_rate_init': [0.001, 0.005]}
stacker = GridSearchCV(MLPClassifier(hidden_layer_sizes=(64, 32, 16), max_iter=500, random_state=42), 
                       mlp_params, cv=3, scoring='f1_macro')
stacker.fit(X_stack_train, y)

ensemble_predictions = stacker.best_estimator_.predict_proba(X_stack_test)

# 6. Final Results Visualization
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y, stacker.predict(X_stack_train))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis')
plt.title('Training Confusion Matrix (Stacked Model)')
plt.show()

print(f"Final OOF F1-Score: {f1_score(y, stacker.predict(X_stack_train), average='macro'):.4f}")
print(f"Optimized Parameters: {stacker.best_params_}")

CELL 3: SUBMISSION


In [ ]:
# Convert the probability scores into final class labels (0, 1, 2, or 3)
# We use argmax along the rows to select the class with the highest probability
final_label_predictions = ensemble_predictions.argmax(axis=1)

# To ensure the output matches the required Kaggle format, we load the official sample
# We check for both common naming conventions found in the competition data
submission_format_path = '/kaggle/input/comment-category-prediction-challenge/sample_submission.csv'

if not os.path.exists(submission_format_path):
    # Fallback to alternative filename if sample_submission is not present
    submission_format_path = '/kaggle/input/comment-category-prediction-challenge/Sample.csv'

try:
    # Load the template for the submission
    submission_df = pd.read_csv(submission_format_path)
    
    # Verification: Ensure our model's output matches the expected number of test samples
    if len(submission_df) == len(final_label_predictions):
        submission_df['label'] = final_label_predictions
        
        # Save the final results to a CSV file for submission
        submission_df.to_csv('submission.csv', index=False)
        
        print(f"Success: Submission saved with shape {submission_df.shape}")
        print("Preview of final predictions:")
        print(submission_df.head())
    else:
        # This handles cases where the test data might have been split differently
        print(f"Warning: Count mismatch. Model predicted {len(final_label_predictions)} labels "
              f"but the sample format requires {len(submission_df)}.")

except Exception as error:
    print(f"Processing Error: {error}")
    
    # Emergency fallback: Construct a dataframe from the test set IDs directly
    # This ensures we still have a valid file if the sample CSV is inaccessible
    print("Generating submission file using test set IDs as fallback.")
    submission_df = pd.DataFrame({
        'id': test_data['id'], 
        'label': final_label_predictions
    })
    submission_df.to_csv('submission.csv', index=False)

CELL 3A RESULTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# --- 1. Comparative Performance Matrix ---
# Aggregating OOF metrics to demonstrate the "Ensemble Gain"
y_true = train_data['label']
stacker_preds = stacker_search.predict(X_stack_train)

metrics_data = {
    'Expert Model': ['Logistic Regression', 'LightGBM', 'Naive Bayes', 'Stacked Ensemble (Final)'],
    'F1 (Macro)': [
        f1_score(y_true, oof_lr.argmax(axis=1), average='macro'),
        f1_score(y_true, oof_lgbm.argmax(axis=1), average='macro'),
        f1_score(y_true, oof_nb.argmax(axis=1), average='macro'),
        f1_score(y_true, stacker_preds, average='macro')
    ]
}

performance_df = pd.DataFrame(metrics_data)
print("### MODEL COMPARISON SUMMARY ###")
print(performance_df.to_string(index=False))
print("-" * 40)

# --- 2. Detailed Classification Report ---
# Provides class-specific Precision, Recall, and F1
target_names = ['Neutral (0)', 'Toxic (1)', 'Threat (2)', 'Hate (3)']
print("\n### DETAILED CLASS PERFORMANCE (OOF) ###")
print(classification_report(y_true, stacker_preds, target_names=target_names))

# --- 3. Visualization: Dual-Plot Diagnostics ---
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Plot A: Normalized Confusion Matrix (Error Distribution)
cm = confusion_matrix(y_true, stacker_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names, ax=ax[0])
ax[0].set_title('Normalized Confusion Matrix (Recall per Class)')
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')

# Plot B: Expert Contribution (Variance across Folds)
# Comparing the average confidence (max probability) across experts
expert_conf = [np.mean(np.max(oof_lr, axis=1)), 
               np.mean(np.max(oof_lgbm, axis=1)), 
               np.mean(np.max(oof_nb, axis=1))]
ax[1].bar(['LogReg', 'LGBM', 'NB'], expert_conf, color=['#3498db', '#e74c3c', '#2ecc71'])
ax[1].set_title('Average Expert Confidence Level')
ax[1].set_ylabel('Mean Max Probability')
ax[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

